In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder, StandardScaler
import gc

In [2]:
num_features = 86  
num_codes = 5000  
embed_dim = 8     
max_seq_len = 90
train_df = pd.read_parquet('/kaggle/input/competitions/ts-forecasting/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/ts-forecasting/test.parquet')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
stable_num_features = [
    'feature_cg', 'feature_al', 'feature_ce', 'feature_v', 'feature_cf', 
    'feature_a', 'feature_ar', 'feature_as', 'feature_m', 'feature_aq', 
    'feature_l', 'feature_ai', 'feature_bg', 'feature_ad', 'feature_i', 
    'feature_am', 'feature_h', 'feature_z', 'feature_aj', 'feature_r', 
    'feature_af', 'feature_n', 'feature_u', 'feature_y', 'feature_bp', 
    'feature_ao', 'feature_bz', 'feature_j'
]

for col in stable_num_features:
    if train_df[col].isnull().sum() / len(train_df) >= 0.05:
        train_df[col + '_missing_flag'] = train_df[col].isnull().astype(np.float32)
        test_df[col + '_missing_flag'] = test_df[col].isnull().astype(np.float32)
        stable_num_features.append(col + '_missing_flag')
    
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

scaler = StandardScaler()
train_df[stable_num_features] = scaler.fit_transform(train_df[stable_num_features].astype(np.float32))
test_df[stable_num_features] = scaler.transform(test_df[stable_num_features].astype(np.float32))

categorical_cols = ['code', 'sub_code', 'sub_category', 'horizon']
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()

    all_values = pd.concat([train_df[col], test_df[col]]).astype(str).unique()
    le.fit(all_values)
    
    train_df[col + '_encoded'] = le.transform(train_df[col].astype(str))
    test_df[col + '_encoded'] = le.transform(test_df[col].astype(str))
    encoders[col] = le


In [4]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=8):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, encoding_dim) 
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded


input_dim = len(stable_num_features) 
ae_model = AutoEncoder(input_dim=input_dim, encoding_dim=8).to(device)
ae_criterion = nn.MSELoss()
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)

In [5]:
X_train_ae = torch.FloatTensor(train_df[stable_num_features].values).to(device)

ae_model = AutoEncoder(input_dim=len(stable_num_features), encoding_dim=8).to(device)
ae_criterion = nn.MSELoss()
ae_optimizer = torch.optim.AdamW(ae_model.parameters(), lr=1e-3)

ae_model.train()
ae_epochs = 20
ae_batch_size = 4096 

for epoch in range(ae_epochs):
    permutation = torch.randperm(X_train_ae.size()[0])
    for i in range(0, X_train_ae.size()[0], ae_batch_size):
        indices = permutation[i:i+ae_batch_size]
        batch_x = X_train_ae[indices]
        
        encoded, decoded = ae_model(batch_x)
        loss = ae_criterion(decoded, batch_x)
        
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        
    if (epoch + 1) % 5 == 0:
        print(f"Loss: {loss.item():.5f}")

ae_model.eval()


deep_features_train = []
with torch.no_grad():
    for i in range(0, len(X_train_ae), ae_batch_size):
        batch_x = X_train_ae[i:i+ae_batch_size]
        encoded, _ = ae_model(batch_x)
        deep_features_train.append(encoded.cpu().numpy())
deep_features_train = np.vstack(deep_features_train)


X_test_ae = torch.FloatTensor(test_df[stable_num_features].values).to(device)
deep_features_test = []
with torch.no_grad():
    for i in range(0, len(X_test_ae), ae_batch_size):
        batch_x = X_test_ae[i:i+ae_batch_size]
        encoded, _ = ae_model(batch_x)
        deep_features_test.append(encoded.cpu().numpy())
deep_features_test = np.vstack(deep_features_test)


deep_cols = [f'deep_feat_{i}' for i in range(8)]
train_df[deep_cols] = deep_features_train
test_df[deep_cols] = deep_features_test


stable_num_features.extend(deep_cols)

last_days = train_df.groupby('code').tail(max_seq_len - 1).copy()


Loss: 0.16502
Loss: 0.15478
Loss: 0.16899
Loss: 0.18641


In [6]:
max_time = train_df['ts_index'].max()
split_time = max_time - 100
df_train = train_df[train_df['ts_index'] <= split_time].copy()
df_val = train_df[train_df['ts_index'] >= (split_time - max_seq_len + 1)].copy()
print(f"Số dòng Train: {len(df_train)} | Số dòng Valid: {len(df_val)}")

Số dòng Train: 5173860 | Số dòng Valid: 317915


In [7]:
def clean_hedge_fund_data(df):
    
    feature_cols = [col for col in df.columns if col.startswith('feature_')]
    
    df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    df = df.dropna(subset=['y_target', 'weight'])
    
    scaler = StandardScaler()
    df[feature_cols] = scaler.fit_transform(df[feature_cols])
    
    df[feature_cols] = df.groupby('code')[feature_cols].ffill()
    
    df[feature_cols] = df[feature_cols].fillna(0)
    
    lower_bound = df[feature_cols].quantile(0.01)
    upper_bound = df[feature_cols].quantile(0.99)
    df[feature_cols] = df[feature_cols].clip(lower=lower_bound, upper=upper_bound, axis=1)
    
    return df, scaler


In [8]:
class HedgeFundDataset(Dataset):
    def __init__(self, df, seq_len = 10): 
        self.seq_len = seq_len
        feature_cols = [col for col in df.columns if col.startswith('feature_')]
        self.features = df[stable_num_features].values.astype(np.float32)
        self.codes = df['code_encoded'].values.astype(np.int64)
        self.sub_codes = df['sub_code_encoded'].values.astype(np.int64)     
        self.sub_cats = df['sub_category_encoded'].values.astype(np.int64)
        self.horizons = df['horizon_encoded'].values.astype(np.int64)
        self.targets = (df['y_target'].values).astype(np.float32)
        self.weights = df['weight'].values.astype(np.float32)
        self.valid_indices = []
        for i in range(self.seq_len - 1, len(df)):
            if self.codes[i] == self.codes[i - seq_len + 1]:
                self.valid_indices.append(i)
    def __len__(self):
        return len(self.valid_indices)
    def __getitem__(self, idx):
        end_idx = self.valid_indices[idx]
        start_idx = end_idx - self.seq_len + 1
        x_feat = self.features[start_idx : end_idx + 1]
        x_code = self.codes[start_idx : end_idx + 1]
        x_sub_code = self.sub_codes[start_idx : end_idx + 1] 
        x_sub_cat = self.sub_cats[start_idx : end_idx + 1]
        x_horizon = self.horizons[start_idx : end_idx + 1]
        y = self.targets[end_idx]
        w = self.weights[end_idx]
        return torch.tensor(x_feat), torch.tensor(x_code), torch.tensor(x_sub_code), torch.tensor(x_sub_cat), torch.tensor(x_horizon), torch.tensor(y), torch.tensor(w)

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500): 
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class Transformer(nn.Module):
    def __init__(self, num_features, num_codes, num_sub_codes, num_sub_cats, num_horizons, embed_dim, nhead=4, num_layers=2):
        super(Transformer, self).__init__()
        self.code_embedding = nn.Embedding(num_codes, embed_dim)
        self.feature_projection = nn.Linear(num_features, embed_dim)
        self.sub_code_embedding = nn.Embedding(num_sub_codes, embed_dim) 
        self.sub_cat_embedding = nn.Embedding(num_sub_cats, embed_dim)
        self.horizon_embedding = nn.Embedding(num_horizons, embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.skip_linear = nn.Linear(num_features, 1)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model = embed_dim, nhead = nhead, batch_first = True, norm_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Linear(embed_dim, 1)
        self.combine_layer = nn.Linear(embed_dim * 5, embed_dim)
        self.emb_layer_norm = nn.LayerNorm(embed_dim)
    def forward(self, x_features, x_code, x_sub_code, x_sub_cat, x_horizon):
        feat_emb = self.feature_projection(x_features)
        code_emb = self.code_embedding(x_code)
        sub_code_emb = self.sub_code_embedding(x_sub_code) 
        sub_cat_emb = self.sub_cat_embedding(x_sub_cat)
        horz_emb = self.horizon_embedding(x_horizon)
        x = torch.cat([feat_emb, code_emb, sub_code_emb, sub_cat_emb, horz_emb], dim=-1)
        x = self.combine_layer(x) 
        x = torch.nn.functional.gelu(x)
        x = self.emb_layer_norm(x)
        x = self.dropout(x)
        x = self.pos_encoder(x)
        output = self.transformer_encoder(x)
        last_step = output[:, -1, :]
        deep_pred = self.decoder(last_step)
        wide_pred = self.skip_linear(x_features[:, -1, :])
        return deep_pred + wide_pred


In [10]:
class LSTMModel(nn.Module):
    def __init__(self, num_features, num_codes, num_sub_codes, num_sub_cats, num_horizons, embed_dim=64, hidden_size=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.feature_projection = nn.Linear(num_features, embed_dim)
        self.code_embedding = nn.Embedding(num_codes, embed_dim)
        self.sub_code_embedding = nn.Embedding(num_sub_codes, embed_dim) 
        self.sub_cat_embedding = nn.Embedding(num_sub_cats, embed_dim)
        self.horizon_embedding = nn.Embedding(num_horizons, embed_dim)
        combined_dim = embed_dim * 5
        
        self.lstm = nn.LSTM(
            input_size=combined_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,    
            dropout=0.2 if num_layers > 1 else 0.0
        )
        self.decoder = nn.Linear(hidden_size, 1)

    def forward(self, x_features, x_code, x_sub_code, x_sub_cat, x_horizon):
        feat_emb = self.feature_projection(x_features)
        code_emb = self.code_embedding(x_code)
        sub_code_emb = self.sub_code_embedding(x_sub_code) 
        sub_cat_emb = self.sub_cat_embedding(x_sub_cat)
        horz_emb = self.horizon_embedding(x_horizon)
        
        x = torch.cat([feat_emb, code_emb, sub_code_emb, sub_cat_emb, horz_emb], dim=-1)
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_step = lstm_out[:, -1, :]
        return self.decoder(last_step)

In [11]:
train_dataset = HedgeFundDataset(df_train, seq_len=max_seq_len)
val_dataset = HedgeFundDataset(df_val, seq_len=max_seq_len)
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=1024,      
    shuffle=True, 
    drop_last=True,       
    num_workers=4,        
    pin_memory=True       
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=1024,      
    shuffle=False,       
    drop_last=False,      
    num_workers=4,        
    pin_memory=True       
)
max_code_id = int(max(train_df['code_encoded'].max(), test_df['code_encoded'].max()) + 1)
max_sub_code_id = int(max(train_df['sub_code_encoded'].max(), test_df['sub_code_encoded'].max()) + 1)
max_sub_cat_id = int(max(train_df['sub_category_encoded'].max(), test_df['sub_category_encoded'].max()) + 1)
max_horizon_id = int(max(train_df['horizon_encoded'].max(), test_df['horizon_encoded'].max()) + 1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = LSTMModel(
#     num_features = len(stable_num_features), 
#     num_codes = max_code_id, 
#     num_sub_codes = max_sub_code_id, 
#     num_sub_cats = max_sub_cat_id, 
#     num_horizons = max_horizon_id,
#     embed_dim = 64,
#     hidden_size = 128,   
#     num_layers = 2
# ).to(device)
class Loss(nn.Module):
    def __init__(self, eps=1e-8, alpha=0.6):
        super().__init__()
        self.eps = eps 
        self.alpha = alpha 
    def forward(self, y_pred, y_target, weights):
        y_pred = y_pred.view(-1)
        y_target = y_target.view(-1)
        weights = weights.view(-1)


        numerator = torch.sum(weights * (y_target - y_pred) ** 2)
        denominator = torch.sum(weights * (y_target ** 2))
        mse_loss = numerator / (denominator + self.eps)


        pred_mean = torch.mean(y_pred)
        target_mean = torch.mean(y_target)
        pred_std = torch.std(y_pred) + self.eps
        target_std = torch.std(y_target) + self.eps

        covariance = torch.mean((y_pred - pred_mean) * (y_target - target_mean))
        pearson_corr = covariance / (pred_std * target_std)

        pearson_loss = 1.0 - pearson_corr

        return self.alpha * mse_loss + (1 - self.alpha) * pearson_loss

class Loss2(nn.Module):
    def __init__(self, eps=1e-8, mse_weight=0.2):
        super().__init__()
        self.eps = eps
        self.mse_weight = mse_weight

    def forward(self, y_pred, y_target, weights):
        y_pred = y_pred.view(-1)
        y_target = y_target.view(-1)
        weights = weights.view(-1)

        w_norm = weights / (torch.sum(weights) + self.eps)

        pred_mean = torch.sum(w_norm * y_pred)
        target_mean = torch.sum(w_norm * y_target)
        
        covariance = torch.sum(w_norm * (y_pred - pred_mean) * (y_target - target_mean))
        pred_var = torch.sum(w_norm * (y_pred - pred_mean) ** 2)
        target_var = torch.sum(w_norm * (y_target - target_mean) ** 2)
        
        pred_std = torch.sqrt(pred_var + self.eps)
        target_std = torch.sqrt(target_var + self.eps)
        
        pearson_corr = covariance / (pred_std * target_std)
        pearson_loss = 1.0 - pearson_corr

        mse_loss = torch.sum(w_norm * (y_pred - y_target)**2)

        return (1.0 - self.mse_weight) * pearson_loss + self.mse_weight * mse_loss

In [12]:
model = Transformer(
   num_features = len(stable_num_features),
   num_codes = max_code_id,
   num_sub_codes = max_sub_code_id,       
   num_sub_cats = max_sub_cat_id,
   num_horizons = max_horizon_id,
   embed_dim = 64
).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=0)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [13]:
import copy
epochs = 4
criterion = Loss()
best_val_loss = float('inf')
best_model_weights = None

for epoch in range(epochs):
    model.train() 
    total_train_loss = 0.0
    
    for batch_idx, (x_feat, x_code, x_sub_code, x_sub_cat, x_horizon, y, w) in enumerate(train_dataloader):
        x_feat, x_code, x_sub_code, x_sub_cat, x_horizon, y, w = x_feat.to(device), x_code.to(device), x_sub_code.to(device), x_sub_cat.to(device), x_horizon.to(device), y.to(device), w.to(device)
        
        optimizer.zero_grad()
        predictions = model(x_feat, x_code, x_sub_code, x_sub_cat, x_horizon)
        loss = criterion(predictions, y, w)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Train Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    
    model.eval() 
    total_val_loss = 0.0
    
    with torch.no_grad(): 
        for val_batch_idx, (x_feat, x_code, x_sub_code, x_sub_cat, x_horizon, y, w) in enumerate(val_dataloader):
            x_feat, x_code, x_sub_code, x_sub_cat, x_horizon, y, w = x_feat.to(device), x_code.to(device), x_sub_code.to(device), x_sub_cat.to(device), x_horizon.to(device), y.to(device), w.to(device)
            
            predictions = model(x_feat, x_code, x_sub_code, x_sub_cat, x_horizon)
            loss = criterion(predictions, y, w)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)
    
    print(f"Hết Epoch {epoch+1} | Trung bình Train Loss: {avg_train_loss:.4f} | Trung bình Val Loss: {avg_val_loss:.4f}")
    
    if 'scheduler' in locals():
        scheduler.step(avg_val_loss)
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_weights = copy.deepcopy(model.state_dict())

model.load_state_dict(best_model_weights)

Epoch 1 | Batch 0 | Train Loss: 101339.6484
Epoch 1 | Batch 100 | Train Loss: 9224.5859
Epoch 1 | Batch 200 | Train Loss: 5545.6509
Epoch 1 | Batch 300 | Train Loss: 2352.6775
Epoch 1 | Batch 400 | Train Loss: 884.0645
Epoch 1 | Batch 500 | Train Loss: 343.7029
Epoch 1 | Batch 600 | Train Loss: 158.8644
Epoch 1 | Batch 700 | Train Loss: 60.3865
Epoch 1 | Batch 800 | Train Loss: 270.8095
Epoch 1 | Batch 900 | Train Loss: 21.8632
Epoch 1 | Batch 1000 | Train Loss: 30.1355
Epoch 1 | Batch 1100 | Train Loss: 56.1171
Epoch 1 | Batch 1200 | Train Loss: 27.4483
Epoch 1 | Batch 1300 | Train Loss: 16.3769
Epoch 1 | Batch 1400 | Train Loss: 27.2093
Epoch 1 | Batch 1500 | Train Loss: 20.4324
Epoch 1 | Batch 1600 | Train Loss: 17.0162
Epoch 1 | Batch 1700 | Train Loss: 11.6618
Epoch 1 | Batch 1800 | Train Loss: 24.8749
Epoch 1 | Batch 1900 | Train Loss: 28.7660
Epoch 1 | Batch 2000 | Train Loss: 14.9973
Epoch 1 | Batch 2100 | Train Loss: 18.2491
Epoch 1 | Batch 2200 | Train Loss: 17.0601
Epoch 1 |

<All keys matched successfully>

In [14]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

test_ids = test_df['id'].tolist()


last_days['is_test'] = False
test_df['is_test'] = True

df_combined = pd.concat([last_days, test_df], ignore_index=True)
df_combined = df_combined.sort_values(by=['code', 'ts_index']).reset_index(drop=True)

# df_combined[stable_num_features] = scaler.transform(df_combined[stable_num_features])
df_combined[stable_num_features] = df_combined.groupby('code')[stable_num_features].ffill()
df_combined[stable_num_features] = df_combined[stable_num_features].fillna(0)

categorical_cols = ['code', 'sub_code', 'sub_category', 'horizon']

for col in categorical_cols:

    mapping = {val: i for i, val in enumerate(encoders[col].classes_)}
    
    df_combined[col + '_encoded'] = df_combined[col].astype(str).map(mapping).fillna(0).astype(int)

class TestHedgeFundDataset(Dataset):
    def __init__(self, df, seq_len):
        self.seq_len = seq_len
        
        self.features = df[stable_num_features].values.astype(np.float32)
        
        self.codes = df['code_encoded'].values.astype(np.int64) 
        self.sub_codes = df['sub_code_encoded'].values.astype(np.int64)
        self.sub_cats = df['sub_category_encoded'].values.astype(np.int64)
        self.horizons = df['horizon_encoded'].values.astype(np.int64)
        self.is_test = df['is_test'].values 
        self.row_ids = df['id'].values
        
        self.valid_indices = []
        for i in range(self.seq_len - 1, len(df)):
            if self.codes[i] == self.codes[i - self.seq_len + 1] and self.is_test[i]:
                self.valid_indices.append(i)

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        end_idx = self.valid_indices[idx]
        start_idx = end_idx - self.seq_len + 1
        
        x_feat = self.features[start_idx : end_idx + 1]
        x_code = self.codes[start_idx : end_idx + 1]
        x_sub_code = self.sub_codes[start_idx : end_idx + 1]
        x_sub_cat = self.sub_cats[start_idx : end_idx + 1]
        x_horizon = self.horizons[start_idx : end_idx + 1]
        
        return (torch.tensor(x_feat), torch.tensor(x_code), torch.tensor(x_sub_code), 
                torch.tensor(x_sub_cat), torch.tensor(x_horizon), self.row_ids[end_idx])
        
test_dataset = TestHedgeFundDataset(df_combined, seq_len=max_seq_len)

In [15]:
test_dataloader = DataLoader(
    test_dataset, 
    batch_size=1024,      
    shuffle=False,        
    drop_last=False,   
    num_workers=2,        
    pin_memory=True      
)
def generate_submission(model, dataloader, test_ids):
    model.eval()
    pred_dict = {}
    
    with torch.no_grad():
        for x_feat, x_code, x_sub_code, x_sub_cat, x_horizon, batch_ids in dataloader:
            
            x_feat = x_feat.to(device)
            x_code = x_code.to(device)
            x_sub_code = x_sub_code.to(device)
            x_sub_cat = x_sub_cat.to(device)
            x_horizon = x_horizon.to(device)
            
            out = model(x_feat, x_code, x_sub_code, x_sub_cat, x_horizon)
            preds_numpy = out.cpu().numpy() 
            
            for i, row_id in enumerate(batch_ids):
                pred_dict[row_id] = preds_numpy[i][0]

    final_predictions = []
    missing_count = 0
    
    for tid in test_ids:
        if tid in pred_dict:
            final_predictions.append(pred_dict[tid])
        else:
            final_predictions.append(0.0)
            missing_count += 1
            
    submission = pd.DataFrame({
        'id': test_ids, 
        'prediction': final_predictions
    })
    
    submission.to_csv('submission.csv', index=False)
    return submission

submission_df = generate_submission(model, test_dataloader, test_ids)

In [16]:
submission_df

,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.016237
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.015855
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.015484
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.015316
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.020638
...,...,...
1447102,83EG83KQ__VYN97209__PHHHVYZI__3__4305,-0.012649
1447103,83EG83KQ__VYN97209__PHHHVYZI__1__4306,-0.012293
1447104,83EG83KQ__VYN97209__PHHHVYZI__3__4306,-0.012849
1447105,83EG83KQ__VYN97209__PHHHVYZI__1__4307,-0.011941


In [17]:
import lightgbm as lgb
import gc
import pandas as pd

encoded_cat_features = ['code_encoded', 'sub_code_encoded', 'sub_category_encoded', 'horizon_encoded']

training_features = encoded_cat_features + ['ts_index'] + stable_num_features

max_time = df_train['ts_index'].max()
split_time = max_time - 60


X_train = df_train[df_train['ts_index'] <= split_time][training_features]
y_train = df_train[df_train['ts_index'] <= split_time]['y_target']
w_train = df_train[df_train['ts_index'] <= split_time]['weight']


X_val = df_train[df_train['ts_index'] > split_time][training_features]
y_val = df_train[df_train['ts_index'] > split_time]['y_target']
w_val = df_train[df_train['ts_index'] > split_time]['weight']

train_data = lgb.Dataset(X_train, label=y_train, weight=w_train, categorical_feature=encoded_cat_features)
val_data = lgb.Dataset(X_val, label=y_val, weight=w_val, reference=train_data, categorical_feature=encoded_cat_features)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,       
    'num_leaves': 128,           
    'max_depth': 8,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,
    'min_child_samples': 50,     
    'verbose': -1,
    'random_state': 42,
    'lambda_l1': 0.5,
    'lambda_l2': 0.5
}

model_lgb = lgb.train(
    params,
    train_data,
    num_boost_round=1500,        
    valid_sets=[train_data, val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(50)]
)

X_test = test_df[training_features]

test_preds = model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration)

test_preds_dampened = test_preds * 0.5 

submission_lgb = pd.DataFrame({
    'id': test_df['id'],
    'prediction': test_preds_dampened
})

submission_lgb.to_csv('submission_lgbm.csv', index=False)

Training until validation scores don't improve for 100 rounds
[50]	training's rmse: 0.00205051	valid_1's rmse: 0.00421162
[100]	training's rmse: 0.00201737	valid_1's rmse: 0.00427323
Early stopping, best iteration is:
[16]	training's rmse: 0.00209939	valid_1's rmse: 0.00387189


In [18]:
submission_lgb

,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.002431
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.014957
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.020306
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.001765
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.016334
...,...,...
1447102,83EG83KQ__VYN97209__PHHHVYZI__3__4305,-0.000012
1447103,83EG83KQ__VYN97209__PHHHVYZI__1__4306,-0.000012
1447104,83EG83KQ__VYN97209__PHHHVYZI__3__4306,-0.000012
1447105,83EG83KQ__VYN97209__PHHHVYZI__1__4307,-0.000012
